<a href="https://colab.research.google.com/github/lcbjrrr/DBMS/blob/main/KeyPairDBs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://pbs.twimg.com/media/HQQD9CdXwAEfOSN?format=jpg&name=large)


**Transactional SQL**: Relational database systems prioritizing strict ACID compliance, fixed schemas, and complex multi-table joins within a single geographical location or region. Regional e-commerce ordering platforms, localized accounting systems, or enterprise HR applications where transactional integrity is critical, but traffic remains concentrated within one region. ***(MySQL / SQLite)***

*Global Scalability*- Worldwide core-banking networks, international flight reservation systems, or global ledger applications where double-spending or overbooking across continents must be strictly prevented. ***(Postgres)***


**Transactional NoSQL**: Dynamic, schema-flexible document or key-value stores engineered for high-concurrency, low-latency CRUD operations and seamless client-side data synchronization. Mobile application backends, real-time gaming state managers, active session stores, and dynamic user profile catalogs that require fast reads and writes without complex table relationships. ***(MongoDB)***


**Analytical SQL**: Columnar, massively parallel processing (MPP) data warehouses optimized for executing complex analytical queries, aggregations, and scanning vast historical datasets using standard SQL syntax. Enterprise business intelligence, cross-departmental reporting dashboards, customer churn analysis, and multi-year financial trend forecasting. ***(Iceberg / Databricks)***


**Analytical NoSQL**: High-throughput wide-column or key-value engines built to handle massive, continuous write streams and fast, sequential range scans on append-heavy data. Internet of Things (IoT) sensor telemetry, real-time user clickstream tracking, high-frequency financial ticker ingestion, and operational log monitoring. ***(LevelDB)***


## Casandra

Register to Astra

![](https://pbs.twimg.com/media/HQP6VmKW0AA3L5J?format=jpg&name=small)

Activate you initial DB

![](https://pbs.twimg.com/media/HQP6jyqW4AARkQd?format=jpg&name=small)

Generate the token

![](https://pbs.twimg.com/media/HQP6rrGWMAArD11?format=jpg&name=small)

Download the bundle

![](https://pbs.twimg.com/media/HQP6y1NXYAQWn1N?format=png&name=small)

In [6]:
!pip install cassandra-driver

In [7]:
!pip install finnhub-python

In [ ]:
import sys
import subprocess
import time
import json
import finnhub

from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
SPACE='myspace'
cloud_config = {'secure_connect_bundle': './secure-connect_mydb.zip'}
auth_provider = PlainTextAuthProvider('token',ASTRA)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()
print('Connected to Astra DB')

finnhub_client = finnhub.Client(api_key=APIKEY)

session.set_keyspace(SPACE)
session.execute("CREATE TABLE IF NOT EXISTS quotes (ts bigint PRIMARY KEY,data text);")
insert_statement = session.prepare("INSERT INTO quotes (ts, data) VALUES (?, ?)")

ITS = 2222
TIM = 1
print("Starting data collection and storage to Cassandra...")
for i in range(ITS):
    try:
        # Make the API call
        quote_data = finnhub_client.quote('BINANCE:DOGEUSDT')
        if quote_data and 't' in quote_data:
            timestamp = quote_data['t'] # 't' is already an integer (Unix timestamp)
            # Create a dictionary for the value, excluding 't'
            value_data = {k: v for k, v in quote_data.items() if k != 't'}
            # Serialize the value dictionary to JSON string
            value_json_string = json.dumps(value_data)
            # Insert into Cassandra using the prepared statement
            session.execute(insert_statement, [timestamp, value_json_string])
            print(f"[{i+1}/{ITS}] Stored data for timestamp: {timestamp} in Cassandra: {value_json_string}")
        else:
            print(f"[{i+1}/{ITS}] No 't' key found in quote data or quote data is empty. Skipping this iteration.")
    except Exception as e:
        print(f"[{i+1}/{ITS}] An error occurred during API call or Cassandra operation: {e}")
    # Sleep for TIM seconds before the next call
    time.sleep(TIM)
print("Data collection complete to Cassandra.")

# Don't forget to close the cluster connection when you are completely done with it.
#cluster.shutdown()

Connected to Astra DB
Starting data collection and storage to Cassandra...
[1/2222] Stored data for timestamp: 1787322797 in Cassandra: {"c": 0.08384, "d": 0.00572, "dp": 7.3221, "h": 0.08577, "l": 0.07763, "o": 0.07812, "pc": 0.07812}
[2/2222] Stored data for timestamp: 1787322797 in Cassandra: {"c": 0.08384, "d": 0.00572, "dp": 7.3221, "h": 0.08577, "l": 0.07763, "o": 0.07812, "pc": 0.07812}
[3/2222] Stored data for timestamp: 1787322797 in Cassandra: {"c": 0.08384, "d": 0.00572, "dp": 7.3221, "h": 0.08577, "l": 0.07763, "o": 0.07812, "pc": 0.07812}
[4/2222] Stored data for timestamp: 1787322797 in Cassandra: {"c": 0.08384, "d": 0.00572, "dp": 7.3221, "h": 0.08577, "l": 0.07763, "o": 0.07812, "pc": 0.07812}
[5/2222] Stored data for timestamp: 1787322797 in Cassandra: {"c": 0.08384, "d": 0.00572, "dp": 7.3221, "h": 0.08577, "l": 0.07763, "o": 0.07812, "pc": 0.07812}
[6/2222] Stored data for timestamp: 1787322797 in Cassandra: {"c": 0.08384, "d": 0.00572, "dp": 7.3221, "h": 0.08577, "l

CQL

![](https://pbs.twimg.com/media/HQQMhugX0AA6C5_?format=jpg&name=large)

In [13]:
search_id = 1787321251
start_time = time.perf_counter()
select_statement = session.prepare("SELECT ts, data FROM quotes WHERE ts = ?")
print(f"Searching for record with ts = {search_id}...")
try:
    result_set = session.execute(select_statement, [search_id])
    if result_set:
        for row in result_set:
            print(f"Found record: timestamp={row.ts}, data={row.data}")
    else:
        print(f"No record found for ts = {search_id}")
except Exception as e:
    print(f"An error occurred during the search operation: {e}")
elapsed_time = time.perf_counter() - start_time
print(elapsed_time*1000,' ms')

Searching for record with ts = 1787321251...
Found record: timestamp=1787321251, data={"c": 0.08384, "d": 0.00624, "dp": 8.0412, "h": 0.08577, "l": 0.07755, "o": 0.0776, "pc": 0.0776}
281.7777889995341  ms


In [11]:
import json
import time
THV = 0.083
start_time = time.perf_counter()
session.set_keyspace(SPACE)

try:
    rows = session.execute(f"SELECT ts, data FROM quotes ALLOW FILTERING")
    found_records = []
    for row in rows:
        try:
            data_dict = json.loads(row.data)
            data_float = float(data_dict['c'])
            if data_float < THV:
                found_records.append({'ts': row.ts, 'value': data_float})
        except json.JSONDecodeError:
            print(f"Error.: {row.ts}: {row.data}")
        except KeyError:
            print(f"Error..: {row.ts}: {row.data}")
        except ValueError:
            print(f"Error..: {row.ts}: {data_dict.get('c', 'N/A')}")
    if found_records:
        print(f"Found {len(found_records)} lower than {THV}:")
        for record in found_records:
            print(f"  Timestamp: {record['ts']}, Value: {record['value']}")
    else:
        print(f"No records lower than {THV}.")
except Exception as e:
    print(f"ERROR: {e}")
elapsed_time = time.perf_counter() - start_time
print(elapsed_time*1000,' ms')


Found 15 lower than 0.083:
  Timestamp: 1787319260, Value: 0.08295
  Timestamp: 1787319241, Value: 0.0829
  Timestamp: 1787319379, Value: 0.08281
  Timestamp: 1787319298, Value: 0.08295
  Timestamp: 1787319143, Value: 0.08274
  Timestamp: 1787319339, Value: 0.08287
  Timestamp: 1787319318, Value: 0.08292
  Timestamp: 1787319279, Value: 0.08299
  Timestamp: 1787319398, Value: 0.08288
  Timestamp: 1787319163, Value: 0.0828
  Timestamp: 1787319182, Value: 0.08292
  Timestamp: 1787319359, Value: 0.08278
  Timestamp: 1787319123, Value: 0.08293
  Timestamp: 1787319202, Value: 0.0828
  Timestamp: 1787319221, Value: 0.0829
252.45423000023948  ms


## LevelDB



In [ ]:
! apt-get update
! apt-get install -y build-essential libleveldb-dev

In [ ]:
!pip install plyvel

In [ ]:
!pip install finnhub-python

In [ ]:
import finnhub
finnhub_client = finnhub.Client(api_key=APIKEY)
finnhub_client.quote('BINANCE:DOGEUSDT')['c']

In [ ]:
import sys
import subprocess
import time
import json
import finnhub
import plyvel
import os

# Initialize Finnhub client using the APIKEY available in the kernel
finnhub_client = finnhub.Client(api_key=APIKEY)

# Initialize LevelDB database
db = plyvel.DB('./LevelDB', create_if_missing=True)
ITS=111
TIM=4
print("Starting data collection and storage...")
for i in range(ITS):
    try:
        # Make the API call
        quote_data = finnhub_client.quote('BINANCE:DOGEUSDT')
        if quote_data and 't' in quote_data:
            # Extract 't' for the key
            key = str(quote_data['t']).encode('utf-8')
            # Create a dictionary for the value, excluding 't'
            value_data = {k: v for k, v in quote_data.items() if k != 't'}
            # Serialize the value dictionary to JSON string and encode to bytes
            value = json.dumps(value_data).encode('utf-8')
            # Insert into LevelDB using db.put()
            db.put(key, value)
            print(f"[{i+1}/{ITS}] Stored data for timestamp: {quote_data['t']}", value)
        else:
            print(f"[{i+1}/{ITS}] No 't' key found in quote data or quote data is empty. Skipping this iteration.")
    except Exception as e:
        print(f"[{i+1}/{ITS}] An error occurred during API call or LevelDB operation: {e}")
    # Sleep for 1 second before the next call
    time.sleep(TIM)
print("Data collection complete.")

In [ ]:
KEY=1787241521
start_time = time.perf_counter()
k = db.get(str(KEY).encode('utf-8'))
if k:
  data = json.loads(k.decode('utf-8'))
  print(f"Value for key {k}:\n{json.dumps(data, indent=2)}")
elapsed_time = time.perf_counter() - start_time
print(elapsed_time*1000,' ms')


In [ ]:
TH=0.082
start_time = time.perf_counter()
found_count=0
for key, value in db.iterator():
    try:
        timestamp = key.decode('utf-8')
        record = json.loads(value.decode('utf-8'))
        if 'c' in record and record['c'] < TH:
            #d=json.dumps(record, indent=2)
            print(f"Found entry with timestamp: {timestamp}",f"  Data: {record['c']}")
            found_count += 1
    except Exception as e:
        print(f"An error occurred processing entry with key {key}: {e}")
print("Entries found meeting the criteria:",found_count)
elapsed_time = time.perf_counter() - start_time
print(elapsed_time*1000,' ms')

## Appendix



```
!pip install websocket-client

#https://pypi.org/project/websocket_client/
import websocket

def on_message(ws, message):
    print(message)

def on_error(ws, error):
    print(error)

def on_close(ws):
    print("### closed ###")

def on_open(ws):
    ws.send('{"type":"subscribe","symbol":"BINANCE:BTCUSDT"}')


websocket.enableTrace(False)
ws = websocket.WebSocketApp("wss://ws.finnhub.io?token=",
                          on_message = on_message,
                          on_error = on_error,
                          on_close = on_close)
ws.on_open = on_open
ws.run_forever()
```





```
import sys
import subprocess
import time
import json
import finnhub

from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
SPACE='spacexx'
cloud_config = {'secure_connect_bundle': './secure-connect-mydb.zip'}
auth_provider = PlainTextAuthProvider('token',ASTRA)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()
print('Connected to Astra DB')

finnhub_client = finnhub.Client(api_key=APIKEY)

session.set_keyspace(SPACE)
session.execute("CREATE TABLE IF NOT EXISTS quotes (ts bigint PRIMARY KEY,data float);")
insert_statement = session.prepare("INSERT INTO quotes (ts, data) VALUES (?, ?)")

ITS = 2
TIM = 11
print("Starting data collection and storage to Cassandra...")
for i in range(ITS):
    try:
        # Make the API call
        quote_data = finnhub_client.quote('BINANCE:DOGEUSDT')
        if quote_data and 't' in quote_data:
            timestamp = quote_data['t'] # 't' is already an integer (Unix timestamp)
            # Create a dictionary for the value, excluding 't'
            value_data = {k: v for k, v in quote_data.items() if k != 't'}
            # Serialize the value dictionary to JSON string
            value_json_string = json.dumps(value_data)
            # Insert into Cassandra using the prepared statement
            session.execute(insert_statement, [timestamp, float(quote_data['c'])])
            print(f"[{i+1}/{ITS}] Stored data for timestamp: {timestamp} in Cassandra: {value_json_string}")
        else:
            print(f"[{i+1}/{ITS}] No 't' key found in quote data or quote data is empty. Skipping this iteration.")
    except Exception as e:
        print(f"[{i+1}/{ITS}] An error occurred during API call or Cassandra operation: {e}")
    # Sleep for TIM seconds before the next call
    time.sleep(TIM)
print("Data collection complete to Cassandra.")
# Don't forget to close the cluster connection when you are completely done with it.
#cluster.shutdown()

#### SEARCH
import json
import time

THV = 0.082
start_time = time.perf_counter()
session.set_keyspace(SPACE
try:
    rows = session.execute(f"SELECT ts, data FROM quotes WHERE data<{THV} ALLOW FILTERING")
    found_records = []
    for row in rows:
        try:
            found_records.append({'ts': row.ts, 'c_value': row.data})
        except json.JSONDecodeError:
            print(f"Error:  {row.ts}: {row.data}")
        except KeyError:
            print(f"Error:  {row.ts}: {row.data}")
        except ValueError:
            print(f"Error:  {row.ts}: {data_dict.get('c', 'N/A')}")

    if found_records:
        print(f"Found {len(found_records)} lower than {THV}:")
        for record in found_records:
            print(f"  Timestamp: {record['ts']}, Value 'c': {record['c_value']}")
    else:
        print(f"No records lower than {THV}.")
except Exception as e:
    print(f"ERROR: {e}")
elapsed_time = time.perf_counter() - start_time
print(f"{elapsed_time*1000} ms")
```

